# derived_8.4-eval-mlp-2.3 — final frontier check for the mlp-2.0 architecture: 320²-hubergelu/lr6e-4 cell refinement + the mixed gelu 3-layer at low lr + the 96 lr3e-4 debiased pool (~1.75 h gpu_debug H100 wall)

Follow-up to `derived_8.4-eval-mlp-2.2` (an honest negative on the val-selected winners — below 2.0's 0.8003 — but the series' strongest single MLP on test: the 54-family `w320x320_d0.4_huber0.2_gelu_lr6e-4` → 0.7973, val rank 49/82, invisible to the val selector; the val-year diagnostic showed val-2021 is the reliable proxy for 54/96 while val-2022 is noise; the 54 val top-10 was dominated by 3-layer configs that overfit val and fail on test; the 96-family is only debiased in the lr3e-4 small-net region). 2.3 is an **optimization + further parameter sweep** of the 2.2 frontiers, temporal protocol only (no LOSO, same honest protocol as 2.0/2.1/2.2), **sized to spend ~1.75 h of the 2 h `gpu_debug` H100 wall allocation** (~681 job-seeds at 8 workers ≈ 8.5 GPU-h), asking **"is the mlp-2.0 (2-regime) architecture still meaningful?"**:

1. **Full 3-seed pool (NEW)** — phase-2/3 top-Ns are capped at the deduped family sizes, so EVERY config trains seeds {42, 7, 123}; the 3-seed mean val RMSE and the val-year diagnostic now cover the entire pool (2.2 gave only the phase-3 top-M a 3rd seed).
2. **The 54-family 320²-hubergelu/lr6e-4 frontier refinement** — δ {0.1..0.3} × lr {4e-4..8e-4, 1e-3} × d {0.3, 0.4, 0.5} at 320², widths {256..384} × huber, and the near-unbiased small-net region (128²–256² gelu, lr {3e-4..6e-4}; w192x192_d0.3_gelu_lr4e-4 hit 0.7934 with a bias²/MSE share of 0.02 % in 2.2).
3. **The mixed gelu 3-layer cell at low lr** — 2.2's mixed test-best `w448x448x448_d0.3_huber0.1_gelu` (0.7940, only 2 seeds in 2.2!); 2.3 grids δ {0.05, 0.1, 0.2} × lr {2e-4..5e-4} at {384³, 448³, 512³}, d {0.2, 0.4} probes, and the untested silu-512³/448³ lr3e-4 cells.
4. **The 96-family lr3e-4-only pool** — 2.2 showed the mid-lr/huber/mixup/max_epochs variants are all worse AND more biased (96 median bias²/MSE 21.7 %); the debiased region is lr3e-4 small nets (w256x256_d0.5: 1.1 %). 2.3 gives the 2.2 test-best w256x256_d0.5 (1 seed in 2.2!) its full 3-seed coverage.
5. **No training-path changes** — the mlp23 trainer is byte-identical to mlp22; anchors reproduce 2.2 bit-identically (stack check via `compare_anchor_vs_2.2.py`).

Protocol unchanged and honest: train on train (2017–2020, n=9,803), early-stop/select on official val (2021–2022, n=4,805), evaluate on untouched test (2023–2025, n=6,620); 3-seed mean val RMSE selection among the mlp/fg/plr winner pool (mlp-only in practice); aux2020 diagnostic only; patience-60 kept; no calibration / no trainval retrain (documented negatives). LOSO out of scope (same protocol as 2.0/2.1/2.2).

All numbers below are the stdout of this executed notebook. Weights/checkpoints/test predictions under `models/`; preprocessed tensors and per-job logs under `artifacts/`; figures at the experiment root.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Robust resolution of the experiment dir whether executed from notebooks/ or in-place.
candidates = [Path.cwd() / "experiment/derived_8.4-eval-mlp-2.3", Path.cwd()]
EXP_DIR = next((p for p in candidates if (p / "metrics_summary.csv").exists()), Path.cwd())

df_summary = pd.read_csv(EXP_DIR / "metrics_summary.csv")
df_per_regime = pd.read_csv(EXP_DIR / "per_regime_metrics_summary.csv")
df_sweep = pd.read_csv(EXP_DIR / "sweep_results.csv")
df_timing = pd.read_csv(EXP_DIR / "timing_summary.csv")
df_bias = pd.read_csv(EXP_DIR / "bias_summary.csv") if (EXP_DIR / "bias_summary.csv").exists() else None
df_bias_cl = pd.read_csv(EXP_DIR / "bias_by_cluster.csv") if (EXP_DIR / "bias_by_cluster.csv").exists() else None
df_ood = pd.read_csv(EXP_DIR / "ood_summary.csv") if (EXP_DIR / "ood_summary.csv").exists() else None
df_stop22 = pd.read_csv(EXP_DIR / "stopping_22_summary.csv") if (EXP_DIR / "stopping_22_summary.csv").exists() else None
df_stop22_agg = pd.read_csv(EXP_DIR / "stopping_22_aggregate.csv") if (EXP_DIR / "stopping_22_aggregate.csv").exists() else None
df_sel = pd.read_csv(EXP_DIR / "selection_summary.csv") if (EXP_DIR / "selection_summary.csv").exists() else None
df_valyears = pd.read_csv(EXP_DIR / "val_year_summary.csv") if (EXP_DIR / "val_year_summary.csv").exists() else None
with open(EXP_DIR / "selected_features.json") as f:
    selected_meta = json.load(f)
with open(EXP_DIR / "timing_log.json") as f:
    timing_log = json.load(f)

fam_labels = {"2regime_96": "2-Regime-96", "2regime_54": "2-Regime-54", "2regime_mixed": "2-Regime-Mixed"}
print("loaded", len(df_summary), "leaderboard rows,", len(df_sweep), "sweep rows")


loaded 84 leaderboard rows, 223 sweep rows


## Selection Protocol v10 Diagnostic

Selection = multi-seed mean val RMSE among the honest architectures (mlp / fg / plr — 2.3 runs only mlp configs, so the pool is mlp-only in practice). 2.3 introduces the **full 3-seed pool**: phase-2/3 top-Ns are capped at the deduped family sizes, so EVERY config trains seeds {42, 7, 123}. This is the direct mitigation for 2.2's findings — the 54-family val ranking was noisy even at 3 seeds (Spearman(val, test) −0.555 in 2.1 → +0.582 in 2.2, but the 2→3-seed flip still moved the 54 winner to a worse-test config), the 54 val top-10 was dominated by 3-layer val-overfitters (removed from the 2.3 pool), and the 3-seed diagnostics previously covered only the phase-3 subset. aux2020 stays diagnostic-only (measures train fit). This section reports the val ranking, the Spearman correlations vs test at 1-/2-/3-seed aggregation over the FULL pool, and the phase-stability table from `analyze_selection.py`.


In [2]:
from scipy.stats import spearmanr
HONEST = ("mlp", "fg", "plr")
print("### Selection Protocol v10 Diagnostic (selection = multi-seed mean val RMSE; mlp/fg/plr pool)")
for family, fam_label in fam_labels.items():
    sub = df_sweep[df_sweep["family"] == family].dropna(subset=["test_r2"]).copy()
    if sub.empty:
        continue
    sub = sub.sort_values("val_rmse", na_position="last").reset_index(drop=True)
    print(f"\n#### {fam_label} — top-10 by val RMSE")
    cols = ["config_id", "architecture", "n_seeds", "val_rmse", "aux_rmse", "test_r2", "test_rmse", "test_bias"]
    print(sub.head(10)[cols].to_markdown(index=False))
    valid = sub.dropna(subset=["val_rmse", "test_r2"])
    if len(valid) >= 8:
        rho, p = spearmanr(valid["val_rmse"], valid["test_r2"])
        print(f"  Spearman(val_rmse, test_r2) = {rho:+.3f} (p={p:.3f}, n={len(valid)})")
    honest_sub = sub[sub["architecture"].isin(HONEST)]
    val_honest = honest_sub.sort_values("val_rmse").iloc[0]
    test_best = sub.sort_values("test_r2", ascending=False).iloc[0]
    print(f"  val winner (honest) : {val_honest['config_id']} (test_r2={val_honest['test_r2']:.4f})")
    print(f"  test best (ref)     : {test_best['config_id']} (test_r2={test_best['test_r2']:.4f})")

if df_sel is not None:
    print("\n### Selection-reliability summary (analyze_selection.py)")
    print("#### Spearman(val, test) by aggregation depth (full 3-seed pool)")
    sub = df_sel[df_sel["aggregation"].str.startswith(("1-seed", "2-seed", "3-seed"))]
    print(sub.to_markdown(index=False))
    print("\n#### Phase stability — winner at each seed depth")
    print(df_sel[df_sel["aggregation"].str.startswith("winner|")].to_markdown(index=False))


### Selection Protocol v10 Diagnostic (selection = multi-seed mean val RMSE; mlp/fg/plr pool)

#### 2-Regime-96 — top-10 by val RMSE
| config_id                | architecture   |   n_seeds |   val_rmse |   aux_rmse |   test_r2 |   test_rmse |   test_bias |
|:-------------------------|:---------------|----------:|-----------:|-----------:|----------:|------------:|------------:|
| w512x512x512_d0.3_lr1e-3 | mlp            |         3 |  0.0484652 |  0.0260292 |  0.759483 |   0.0499584 |   0.0188537 |
| w128x128x128_d0.4        | mlp            |         3 |  0.0513085 |  0.0372981 |  0.738032 |   0.0521387 |   0.0198161 |
| w256x256x256_d0.4        | mlp            |         3 |  0.0513375 |  0.0332079 |  0.741311 |   0.0518113 |   0.0228407 |
| w320x320_d0.4            | mlp            |         3 |  0.0518897 |  0.0375998 |  0.734852 |   0.0524542 |   0.02608   |
| w352x352_d0.5            | mlp            |         3 |  0.0520095 |  0.038937  |  0.755707 |   0.0503491 |   0.0215775 |

## Val-Year Selection Reliability (full 3-seed pool)

2.2's headline finding: the official val split's 2022 half is the noise source for the 54/96 families (Spearman(val-2021, test) = +0.747 (96) / +0.454 (54) vs val-2022 = +0.106 / +0.133, both ns), while the mixed family is negatively correlated with test in BOTH years (structural, the c0 = 96-pool half). This diagnostic splits the official val set (2021–2022) by YEAR and asks which val year is the better proxy for test, and whether the val-selected winner is stable under leave-one-val-year-out selection. It is made possible by `val_preds.npy` (best-val predictions per job, saved by the mlp23 trainer — byte-identical to mlp22) + `artifacts/val_meta.npz`. **NEW in 2.3: with the full 3-seed pool the diagnostic covers the ENTIRE config pool at 3-seed aggregation** (2.2's was restricted to the phase-3 subset, n=26–42/family). **Diagnostic only** — the deployed selection rule stays 3-seed mean val RMSE on the FULL official val (protocol unchanged). Backed by `analyze_val_years.py` (`val_year_summary.csv`).


In [3]:
from scipy.stats import spearmanr as _spearmanr
if df_valyears is not None:
    print("### Val-year diagnostic (val-2021 vs val-2022; 3-seed mean val RMSE per config)")
    for family, fam_label in fam_labels.items():
        sub = df_valyears[df_valyears["family"] == family].sort_values("val_rmse", na_position="last").head(10)
        if sub.empty:
            continue
        print(f"\n#### {fam_label} — top-10 by full-val RMSE (with per-year val RMSE)")
        cols = ["config_id", "n_seeds", "val_rmse", "val_2021_rmse", "val_2022_rmse", "test_r2"]
        print(sub[cols].to_markdown(index=False))

    print("\n### Spearman(val signal, test R2) per family (FULL pool, 3-seed aggregation)")
    rows = []
    for family in fam_labels:
        sub = df_valyears[df_valyears["family"] == family].dropna(subset=["val_rmse", "test_r2"])
        for sig in ("val_rmse", "val_2021_rmse", "val_2022_rmse"):
            s = sub.dropna(subset=[sig])
            if len(s) >= 8:
                rho, p = _spearmanr(s[sig], s["test_r2"])
                rows.append({"family": family, "signal": sig, "n_configs": len(s),
                             "spearman": f"{rho:+.3f}", "p_value": f"{p:.4f}"})
    print(pd.DataFrame(rows).to_markdown(index=False))

    print("\n### Winner stability under leave-one-val-year-out selection (3-seed means)")
    rows = []
    for family in fam_labels:
        sub = df_valyears[df_valyears["family"] == family].dropna(subset=["val_rmse"])
        if sub.empty:
            continue
        for sig in ("val_rmse", "val_2021_rmse", "val_2022_rmse"):
            s = sub.dropna(subset=[sig]).sort_values(sig)
            if s.empty:
                continue
            w = s.iloc[0]
            rows.append({"family": family, "selected_by": sig,
                         "winner": w["config_id"], "winner_test_r2": f"{w['test_r2']:.4f}"})
    print(pd.DataFrame(rows).to_markdown(index=False))
else:
    print("val_year_summary.csv not found — run analyze_val_years.py after the sweep.")


### Val-year diagnostic (val-2021 vs val-2022; 3-seed mean val RMSE per config)

#### 2-Regime-96 — top-10 by full-val RMSE (with per-year val RMSE)
| config_id                |   n_seeds |   val_rmse |   val_2021_rmse |   val_2022_rmse |   test_r2 |
|:-------------------------|----------:|-----------:|----------------:|----------------:|----------:|
| w512x512x512_d0.3_lr1e-3 |         3 |  0.0484652 |       0.0403419 |       0.0557868 |  0.759483 |
| w128x128x128_d0.4        |         3 |  0.0513085 |       0.0415644 |       0.0599037 |  0.738032 |
| w256x256x256_d0.4        |         3 |  0.0513375 |       0.0414355 |       0.060066  |  0.741311 |
| w320x320_d0.4            |         3 |  0.0518897 |       0.0449803 |       0.0583087 |  0.734852 |
| w352x352_d0.5            |         3 |  0.0520095 |       0.0448905 |       0.0586049 |  0.755707 |
| w384x384_d0.5            |         3 |  0.0521039 |       0.0457668 |       0.0580459 |  0.764861 |
| w288x288_d0.4            |       

## Overall Model Leaderboard

All evaluated models ranked by pooled test R² over 2023–2025 (6,620 samples, 7 WA stations). MLP rows carry the sweep `config_id` and `n_seeds`; `(val top-k avg)` rows are offline seed-averaged ensembles of the top-k val-selected honest configs (no extra training); `(5-seed champ, ...)` rows are 5-seed champion ensembles of the val-selected winners (extra stability seeds, no trainval retrain — documented negative); `cross-family` rows average the val-selected winners across families. XGBoost rows are the eval-1.1 references; `MLP-1.3` / `MLP-2.0` / `MLP-2.1` / `MLP-2.2` rows are the previous experiments' val-selected winners + test-best references (2.0's mixed val top-5 ensemble 0.8003 is the number 2.3 must beat; 2.2's mixed val winner 0.7809, cross-family 0.7885, and the series-best single test row 0.7973 are the nearer bars); `test-best` rows are reporting only (selection on test would be leakage).


In [4]:
cols = ["model_name", "strategy_name", "pooled_r2", "pooled_rmse", "pooled_ubrmse", "pooled_bias", "pooled_mae", "pooled_pearson"]
print("### Overall Leaderboard (2023-2025 Test Set)")
print(df_summary[cols].to_markdown(index=False))


### Overall Leaderboard (2023-2025 Test Set)
| model_name                                                                                                                                                                 | strategy_name          |   pooled_r2 |   pooled_rmse |   pooled_ubrmse |   pooled_bias |   pooled_mae |   pooled_pearson |
|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-----------------------|------------:|--------------:|----------------:|--------------:|-------------:|-----------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)                                                                                                                                 | XGBoost_Reference      |    0.81496  |     0.0438196 |       0.043337  |    0.00648567 |    0.0337195 |         0.905594 |
| MLP 2-Regime-Mixed (test-best, w384x384x384_d0.3_huber0.1_g

## Hyperparameter Sweep Summary

223 curated phase-1 configs (all `mlp` — `fg`/`plr` are documented negatives from 2.0, `swa` a documented negative from 2.1, 54-family 3-layer a documented 2.2 negative, and none get GPU except the 3 bit-identity anchors + 2 re-check probes) generated deterministically by `make_configs.py` from 2-factor grids around the 2.2 frontiers: 54 320²-hubergelu/lr6e-4 δ × lr × d fine / width × δ × d / small-net mse + huber / silu probes; mixed gelu 3-layer δ × lr @ {384³, 448³, 512³} / d probes / silu lr3e-4 cells; 96 lr3e-4-only width × d / 3-layer / lr2e-4 / me600 probes. 8 parallel H100 workers; **every config trains seeds {42, 7, 123} (full 3-seed pool)**; configs ranked by **multi-seed mean val RMSE** (honest signal); test R² for reference.


In [5]:
for family, fam_label in fam_labels.items():
    sub = df_sweep[df_sweep["family"] == family].sort_values("val_rmse", na_position="last").head(10)
    if sub.empty:
        continue
    print(f"### Sweep Top-10 — {fam_label} (by val RMSE, the honest selection signal)")
    cols = ["config_id", "architecture", "n_seeds", "dropout", "lr", "loss", "val_rmse", "aux_rmse", "test_r2", "test_rmse", "test_bias", "best_epoch", "train_time_s"]
    show = sub[cols].copy()
    show["deployed"] = [json.loads((EXP_DIR / "models" / family / cid / "meta.json").read_text()).get("deployed", "live")
                        if (EXP_DIR / "models" / family / cid / "meta.json").exists() else "" for cid in sub["config_id"]]
    print(show.to_markdown(index=False))
    print()


### Sweep Top-10 — 2-Regime-96 (by val RMSE, the honest selection signal)
| config_id                | architecture   |   n_seeds |   dropout |     lr | loss   |   val_rmse |   aux_rmse |   test_r2 |   test_rmse |   test_bias |   best_epoch |   train_time_s | deployed   |
|:-------------------------|:---------------|----------:|----------:|-------:|:-------|-----------:|-----------:|----------:|------------:|------------:|-------------:|---------------:|:-----------|
| w512x512x512_d0.3_lr1e-3 | mlp            |         3 |       0.3 | 0.001  | mse    |  0.0484652 |  0.0260292 |  0.759483 |   0.0499584 |   0.0188537 |          263 |        130.289 | live       |
| w128x128x128_d0.4        | mlp            |         3 |       0.4 | 0.0003 | mse    |  0.0513085 |  0.0372981 |  0.738032 |   0.0521387 |   0.0198161 |          390 |        202.054 | live       |
| w256x256x256_d0.4        | mlp            |         3 |       0.4 | 0.0003 | mse    |  0.0513375 |  0.0332079 |  0.741311 |   0.

## Per-Regime Performance Breakdown

Cluster 0 holds 73 % of the test rows, so it dominates the pooled R². Per-cluster test metrics for the val top-3 honest configs per family (the mixed family's c1 = 54+10 specialist is expected to hold the ~0.83 R² of the 54-family's c1 while c0 gains the 96-pool fit), the XGBoost references, and the 1.3 / 2.0 / 2.1 / 2.2 reference winners.


In [6]:
print("### Per-Regime Performance Breakdown")
cols = ["strategy_name", "model_name", "cluster", "n_train", "n_test", "r2", "rmse", "ubrmse", "bias", "mae"]
print(df_per_regime[cols].to_markdown(index=False))


### Per-Regime Performance Breakdown
| strategy_name     | model_name                                                   |   cluster |   n_train |   n_test |       r2 |      rmse |    ubrmse |         bias |       mae |
|:------------------|:-------------------------------------------------------------|----------:|----------:|---------:|---------:|----------:|----------:|-------------:|----------:|
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_lr1e-3)                   |         0 |      7156 |     4817 | 0.751308 | 0.0498896 | 0.0471617 |  0.0162712   | 0.0390899 |
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_lr1e-3)                   |         1 |      2647 |     1803 | 0.77822  | 0.0501416 | 0.0430228 |  0.0257531   | 0.0377413 |
| MLP_2regime_96    | MLP 2-Regime-96 (w128x128x128_d0.4)                          |         0 |      7156 |     4817 | 0.706123 | 0.0542328 | 0.0500799 |  0.0208134   | 0.0426266 |
| MLP_2regime_96    | MLP 2-Regime-96 (w128x128x128_d

## Yearly Performance Breakdown

Year-by-year R² on the 2023–2025 test period. 2.0 fixed the historically weak 2025 year for the mixed family's ensembles (2025 R² 0.8336 — best of any model); 2.1's mixed val winner held 2025 at 0.8185; 2.2's winners held 2025 at ~0.81. This table tracks whether the 2.3 winners hold that year.


In [7]:
year_cols = [c for c in df_summary.columns if c.startswith("year_") and c.endswith("_r2")]
print("### Year-by-Year R² Breakdown")
print(df_summary[["model_name", "pooled_r2", *year_cols]].to_markdown(index=False))


### Year-by-Year R² Breakdown
| model_name                                                                                                                                                                 |   pooled_r2 |   year_2023_r2 |   year_2024_r2 |   year_2025_r2 |
|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------------:|---------------:|---------------:|---------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)                                                                                                                                 |    0.81496  |       0.822971 |       0.783256 |       0.83029  |
| MLP 2-Regime-Mixed (test-best, w384x384x384_d0.3_huber0.1_gelu_lr2e-4)                                                                                                     |    0.80135  |       0.788075 |       0.811327 |       0.80011  |
| MLP-2.0 

## Systematic-Bias Diagnostic (headline)

mlp-1.2/1.3 documented the 96-family's systematic positive test bias (bias² ≈ 10–17 % of MSE); 2.0 got the 54-family under 5 %; 2.1 got the 54-family to 0.8 %; 2.2 met the 54 criterion (3.1 %) and improved mixed (12.7 → 9.6 %) but the 96-family WORSENED to 21.7 % — the mid-lr {4e-4, 6e-4, 8e-4} small-net pool is more biased than the lr3e-4 anchor (w256x256_d0.5: 1.1 %). 2.3's 96 pool is therefore **lr3e-4-only by construction** (widths {96..320} × d {0.4, 0.5, 0.6}, 3-layer probes, lr2e-4, me600 at 96²). Success criterion: per-family median bias²/MSE < 5 % for ALL three families. Backed by `analyze_bias.py`.


In [8]:
if df_bias is not None:
    print("### Per-family median bias^2/MSE share (honest architectures)")
    print("| family     | n_configs |   med_bias2_mse_share |   med_test_bias |   med_test_r2 |")
    print("|:-----------|----------:|----------------------:|----------------:|--------------:|")
    for fam, fam_label in fam_labels.items():
        sub = df_bias[(df_bias["family"] == fam) & df_bias["architecture"].isin(HONEST)]
        if sub.empty:
            continue
        print(f"| {fam} | {len(sub)} | {sub['bias2_mse_share'].median():.4f} | "
              f"{sub['test_bias'].median():.4f} | {sub['test_r2'].median():.4f} |")
    cols = ["family", "config_id", "architecture", "test_r2", "test_rmse", "test_bias", "bias2_mse_share"]
    print("\n### Worst 8 configs by bias^2/MSE share (all architectures)")
    print(df_bias.sort_values("bias2_mse_share", ascending=False).head(8)[cols].to_markdown(index=False))
    print("\n### Best 8 configs by bias^2/MSE share (all architectures)")
    print(df_bias.sort_values("bias2_mse_share").head(8)[cols].to_markdown(index=False))
    if df_bias_cl is not None:
        print("\n### Per-cluster median bias^2/MSE share (honest architectures)")
        print("| family     | cluster |   med_bias2_mse_share |   med_test_bias |   med_test_r2 |")
        print("|:-----------|--------:|----------------------:|----------------:|--------------:|")
        for fam, fam_label in fam_labels.items():
            for cl in (0, 1):
                sub = df_bias_cl[(df_bias_cl["family"] == fam) & (df_bias_cl["cluster"] == cl)
                                 & df_bias_cl["architecture"].isin(HONEST)]
                if sub.empty:
                    continue
                print(f"| {fam} | {cl} | {sub['bias2_mse_share'].median():.4f} | "
                      f"{sub['test_bias'].median():.4f} | {sub['test_r2'].median():.4f} |")
else:
    print("bias_summary.csv not found — run analyze_bias.py after the sweep.")


### Per-family median bias^2/MSE share (honest architectures)
| family     | n_configs |   med_bias2_mse_share |   med_test_bias |   med_test_r2 |
|:-----------|----------:|----------------------:|----------------:|--------------:|
| 2regime_96 | 32 | 0.1353 | 0.0183 | 0.7511 |
| 2regime_54 | 145 | 0.0125 | 0.0047 | 0.7870 |
| 2regime_mixed | 46 | 0.0616 | 0.0117 | 0.7871 |

### Worst 8 configs by bias^2/MSE share (all architectures)
| family        | config_id                          | architecture   |   test_r2 |   test_rmse |   test_bias |   bias2_mse_share |
|:--------------|:-----------------------------------|:---------------|----------:|------------:|------------:|------------------:|
| 2regime_96    | w320x320_d0.4                      | mlp            |  0.734852 |   0.0524542 |   0.02608   |          0.247204 |
| 2regime_96    | w128x128_d0.4                      | mlp            |  0.717528 |   0.0541407 |   0.0265355 |          0.240218 |
| 2regime_96    | w96x96_d0.5_me60

## FeatureGroupedMLP / PLR — documented negatives, not re-run in 2.3

2.0 established that the grouped-tower (`fg`, best 0.782) and PLR-encoding (`plr`, best 0.720) architectures underperform the plain MLP (0.790) at this scale — the winning lever was the *feature allocation* (the `2regime_mixed` family), not the tower structure. Per the no-re-spend rule, **2.3 runs no fg/plr configs**; the classes and the validated semantic grouping remain available in `mlp23/feature_groups.py`. The grouping table for the union of the three families' features is printed for reference.


In [9]:
import sys as _sys
_sys.path.insert(0, str(EXP_DIR))
from mlp23.feature_groups import summary_table
union_feats = sorted(set(selected_meta.get("shared_backbone_54", [])) | set(selected_meta.get("candidate_pool_96", [])) | set(selected_meta.get("cluster_1_delta_features", [])))
print(f"Union of the 3 families' features: {len(union_feats)}")
print(summary_table(list(union_feats)))


Union of the 3 families' features: 116
| group_id | group | n_features | features |
|---|---|---|---|
| 0 | smap | 26 | A_d_SMAP_sm_interp_kobs14, A_d_SMAP_sm_interp_kobs30, A_grad_SMAP_sm_interp_kobs14, A_grad_SMAP_sm_interp_kobs30, A_grad_SMAP_sm_interp_kobs7, C_lag_SMAP_sm_interp_kobs12, C_lag_SMAP_sm_interp_kobs30, SMAP_ampm_diff_interp, SMAP_sm_am_interp, SMAP_sm_am_interp_lag1, SMAP_sm_am_interp_lag30, SMAP_sm_am_interp_rollrange30, SMAP_sm_am_interp_rollrange7, SMAP_sm_interp_lag7, SMAP_sm_interp_rollrange30, SMAP_sm_interp_rollrange7, SMAP_sm_pm_interp, SMAP_sm_pm_interp_lag1, SMAP_sm_pm_interp_lag30, SMAP_sm_pm_interp_lag7, SMAP_sm_pm_interp_rollmean30, SMAP_sm_pm_interp_rollrange30, SMAP_sm_pm_interp_rollrange7, V_ema_SMAP_sm_interp_kobs30, V_rollmin_SMAP_sm_interp_kobs14, V_rollmin_SMAP_sm_interp_kobs30 |
| 1 | optical | 7 | A_grad_s2_b11_kobs30, V_rollmin_s2_b11_kobs14, V_rollmin_s2_b11_kobs30, V_rollmin_s2_b12_kobs30, V_rollrng_s2_b11_kobs30, s2_b4, s2_b8 |
| 2 | vegetatio

## Early-Stopping Replay (patience-60 re-check, tag 22)

Offline replay of honest epoch-selection rules on the saved per-epoch curves (`analyze_stopping.py --tag 22`). 1.2/1.3 established that patience-60 is the best honest rule; the `swa_val` rule rows are replayed for completeness (no SWA configs run in 2.3 — the curves are the live ones). 2.3 re-checks patience-60 on the new grids, including the 96-family me600 probes at 96² (does extending the cap help the smallest nets?).


In [10]:
if df_stop22_agg is not None:
    print("### Stopping-rule aggregates (mean pooled test RMSE; lower is better; oracle = unreachable bound)")
    print(df_stop22_agg.to_markdown(index=False))
else:
    print("stopping_22_aggregate.csv not found — run analyze_stopping.py --tag 22 after the sweep.")


### Stopping-rule aggregates (mean pooled test RMSE; lower is better; oracle = unreachable bound)
| family        | rule             |   mean_test_rmse |   median_test_rmse |   n |
|:--------------|:-----------------|-----------------:|-------------------:|----:|
| 2regime_96    | patience60       |        0.0518455 |          0.051905  |  98 |
| 2regime_96    | patience20       |        0.0518455 |          0.051905  |  98 |
| 2regime_96    | patience40       |        0.0518455 |          0.051905  |  98 |
| 2regime_96    | val_aux          |        0.0526654 |          0.052617  |  98 |
| 2regime_96    | swa_val          |        0.0518455 |          0.051905  |  98 |
| 2regime_96    | plateau_w20e1e-4 |        0.0834599 |          0.080948  |  98 |
| 2regime_96    | plateau_w40e1e-4 |        0.0766095 |          0.0730849 |  98 |
| 2regime_96    | plateau_w40e3e-4 |        0.0766095 |          0.0730849 |  98 |
| 2regime_96    | plateau_w60e1e-4 |        0.0669366 |          0.06377

## Extrapolation (OOD) Check

588/6,620 test rows (8.9 %) are OOD on ≥1 top-10 gain feature (same definition as mlp-1.0–2.2). The pure-96 family keeps its OOD strength; the mixed family is in-distribution-strong but OOD-weak (its c1 = 54+10 half carries the 54-family's weak OOD). The 2.3 winners' OOD behavior is reported for the record (family allocation is pinned, so this is a tracking table, not a target).


In [11]:
if df_ood is not None:
    print("### Extrapolation check (OOD test slices)")
    print(df_ood.to_markdown(index=False))
else:
    print("ood_summary.csv not found — run analyze_extrapolation.py after the sweep.")


### Extrapolation check (OOD test slices)
| model                                                   | slice           |    n |       r2 |      rmse |        bias |       mae |
|:--------------------------------------------------------|:----------------|-----:|---------:|----------:|------------:|----------:|
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)               | all             | 6620 | 0.759483 | 0.0499584 |  0.0188537  | 0.0387226 |
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)               | in_distribution | 6032 | 0.753545 | 0.0514519 |  0.0213449  | 0.0401715 |
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)               | ood             |  588 | 0.757848 | 0.0306936 | -0.0067026  | 0.0238591 |
| MLP 2regime-96 (5-seed champ)                           | all             | 6620 | 0.75656  | 0.050261  |  0.0204007  | 0.0391776 |
| MLP 2regime-96 (5-seed champ)                           | in_distribution | 6032 | 0.749983 | 0.0518224 |  0.0227929  | 0.0407259 |
| MLP 2regime-96 (5-

## Overfitting-Symptom Analysis

From the saved artifacts (no retraining), via `analyze_overfitting.py`: train-fit vs held-out gap (aux2020 = train-fit), capacity vs test transfer, and the per-epoch curve shape for each family's val winner. 2.2's winners show the familiar pattern (test min early, val flat, train-fit improving); the 2.3 mitigations in play are the full 3-seed pool, the removal of the 54 3-layer val-overfitters, and the 96 lr3e-4-only pool.


In [12]:
print("### Overfitting symptoms (analyze_overfitting.py)")
# The CLI writes overfitting_summary.csv; recompute the key tables inline.
import sys as _sys
if str(EXP_DIR) not in _sys.path:
    _sys.path.insert(0, str(EXP_DIR))
from analyze_overfitting import compute_overfitting
r = compute_overfitting(df_sweep, EXP_DIR)
print("\n#### 1. Train-fit vs held-out gap (median RMSE)")
print("| family     |   aux2020 (train-fit) |   val |   test |   val/train ratio |")
print("|:-----------|----------------------:|------:|-------:|------------------:|")
for fam in fam_labels:
    print(f"| {fam} | {r[f'{fam}_med_aux']:.4f} | {r[f'{fam}_med_val']:.4f} | "
          f"{r[f'{fam}_med_test']:.4f} | {r[f'{fam}_train_val_ratio']:.1f}x |")
print("\n#### 2. Capacity vs test transfer (median by n_params bucket)")
print("| family     | capacity   |   n_configs |   med_val_rmse |   med_test_r2 |   med_test_bias |")
print("|:-----------|:-----------|------------:|---------------:|--------------:|----------------:|")
for fam in fam_labels:
    for row in r[f"{fam}_capacity"]:
        print(f"| {row['family']} | {row['capacity']} | {row['n_configs']} | "
              f"{row['med_val_rmse']:.4f} | {row['med_test_r2']:.4f} | {row['med_test_bias']:.4f} |")
print("\n#### 3. Per-epoch curve shape for the val winner (cluster-0 specialist)")
print("| family     | config_id |   aux_ep100 |   aux_ep260 |   val_plateau |   test_min |   test_min_epoch |   test_at_best_val |   test_final |   test_rise_after_min |")
print("|:-----------|:----------|------------:|------------:|--------------:|-----------:|-----------------:|-------------------:|-------------:|----------------------:|")
for row in r["curve_rows"]:
    print(f"| {row['family']} | {row['config_id']} | {row['aux_ep100']:.4f} | {row['aux_ep260']:.4f} | "
          f"{row['val_plateau']:.4f} | {row['test_min']:.4f} | {row['test_min_epoch']} | "
          f"{row['test_at_best_val']:.4f} | {row['test_final']:.4f} | {row['test_rise_after_min']:.4f} |")
print("\n#### 4. Systematic bias on test (MLP vs XGBoost references)")
for fam in fam_labels:
    print(f"MLP {fam} median test bias: {r[f'{fam}_med_test_bias']:.4f}")
print("XGBoost references (eval-1.1): 2-regime 0.0065, global 0.0105")


### Overfitting symptoms (analyze_overfitting.py)

#### 1. Train-fit vs held-out gap (median RMSE)
| family     |   aux2020 (train-fit) |   val |   test |   val/train ratio |
|:-----------|----------------------:|------:|-------:|------------------:|
| 2regime_96 | 0.0423 | 0.0535 | 0.0508 | 1.3x |
| 2regime_54 | 0.0263 | 0.0584 | 0.0470 | 2.2x |
| 2regime_mixed | 0.0254 | 0.0503 | 0.0470 | 2.0x |

#### 2. Capacity vs test transfer (median by n_params bucket)
| family     | capacity   |   n_configs |   med_val_rmse |   med_test_r2 |   med_test_bias |
|:-----------|:-----------|------------:|---------------:|--------------:|----------------:|
| 2regime_96 | <200k | 23 | 0.0538 | 0.7496 | 0.0178 |
| 2regime_96 | 200-500k | 8 | 0.0522 | 0.7603 | 0.0201 |
| 2regime_96 | 1M+ | 1 | 0.0485 | 0.7595 | 0.0189 |
| 2regime_54 | <200k | 56 | 0.0597 | 0.7847 | 0.0018 |
| 2regime_54 | 200-500k | 87 | 0.0579 | 0.7898 | 0.0063 |
| 2regime_54 | 500k-1M | 2 | 0.0552 | 0.7636 | 0.0109 |
| 2regime_mixed |

## Timing

The sweep is sized to spend ~1.75 h of the 2 h `gpu_debug` H100 wall allocation: 223 phase-1 × 3 seeds (full 3-seed pool) + 12 champion job-seeds ≈ 681 jobs at 8 parallel workers (2.2's per-seed mean was 43 s at ~5.9 effective workers).


In [13]:
print("### Timing (H100 PCIe 80 GB, 8 parallel workers)")
print(f"Total sweep wall time: {timing_log.get('sweep_wall_s', float('nan')):.1f} s")
print(f"Total training time (all jobs, GPU-seconds): {sum(j.get('train_time_s', 0.0) for j in timing_log.get('jobs', {}).values()):.0f} s")
print(f"Eval wall time: {timing_log.get('eval_wall_s', float('nan')):.1f} s")
print()
slow = sorted(timing_log.get("jobs", {}).items(), key=lambda kv: -(kv[1].get("train_time_s") or 0))[:5]
print("Slowest jobs (3-seed config train_time_s):")
for k, v in slow:
    print(f"  {k:55s} {v.get('train_time_s', float('nan')):8.1f}s  n_seeds={v.get('n_seeds')}")


### Timing (H100 PCIe 80 GB, 8 parallel workers)
Total sweep wall time: 5287.5 s
Total training time (all jobs, GPU-seconds): 31826 s
Eval wall time: 8.7 s

Slowest jobs (3-seed config train_time_s):
  2regime_96/w96x96_d0.5_me600                               260.4s  n_seeds=3
  2regime_96/w96x96_d0.5_me500                               224.7s  n_seeds=3
  2regime_96/w96x96x96_d0.4                                  219.0s  n_seeds=3
  2regime_96/w192x192_d0.4                                   212.1s  n_seeds=3
  2regime_96/w192x192x192_d0.4                               212.1s  n_seeds=3


## Key Takeaways

1. **The mlp-2.0 architecture IS still meaningful — on test.** The mixed test-best `w384x384x384_d0.3_huber0.1_gelu_lr2e-4` → **0.8014** (bias 0.0018) is the first MLP single config above 2.0's mixed val top-5 ensemble (0.8003); the 54 test-best `w320x320_d0.4_huber0.15_gelu_lr5e-4` → **0.7980** (new 54 record, bias 0.0020) and the 96 test-best `w256x256_d0.5_lr2e-4` → **0.7897** (new 96 record, NEGATIVE bias −0.0028) also beat their families' 2.2 records. `test-best` rows are reporting only (selection on test would be leakage) — but the frontier exists: the 2-regime MLP can clear 0.80 as a single model.
2. **Val selection: the mixed family's structural val-noise is GONE — first significant positive Spearman ever.** Full-val +0.318 (p=0.031; 2.1: −0.309, 2.2: −0.413), BOTH val years positive (+0.263 ns / +0.408 p=0.005), and the winner is stable under leave-one-val-year-out. The 2.3 mixed grid (gelu 3-layer at lr {2e-4, 3e-4}) fixed the family's val-proxy problem. The mixed val winner is unchanged (`w512x512x512_d0.3_huber0.03_lr1e-3`, 0.7809) but mixed val top-5 avg rose to 0.7895 (2.2: 0.7850).
3. **54-family val selection did NOT recover: −0.055 (2.2: +0.582) — and the single 3-layer bit-identity anchor still wins the 54 val ranking** (val RMSE 0.0544 vs the best 2-layer 0.0556). The 3-layer val-overfit is structural, not pool-composition luck: even one 3-layer config dominates 54 val selection while failing on test (0.7596). The val-year diagnostic sharpens this: the 2-layer-only pool's val-2022 half is strongly NEGATIVELY correlated with test (−0.472, p=2e-9). The 54 val top-10 avg still improved to 0.7834 (2.2: 0.7770) — the 2-layer ensembles are the real 54 gains.
4. **96 val selection weakened (+0.157 ns vs +0.566)** — val-2021 stays strong (+0.566, p=7e-4) while val-2022 turned negative (−0.316, p=0.078); the big-net `w512x512x512_d0.3_lr1e-3` remains the val winner (stable under all selectors, test 0.7595) while the 96 test-best is the lr2e-4 small net (0.7897). The 3-layer-val-overfit pattern repeats in 96 (w128³/w256³ rank 2–3 on val, test 0.738–0.741).
5. **Debias: 54 met with the best margin yet (median bias²/MSE 1.25 %; w128x128_d0.3_gelu_lr4e-4 bias −1.6e-5); mixed improved to 6.2 % (2.2: 9.6 %) but still >5 %; 96 improved to 13.5 % (2.2: 21.7 %) but unmet** — within 96, the d0.4 variants are the biased cells (0.20–0.25 share) while lr2e-4 goes negative-bias.
6. **Reproducibility:** 3/3 anchors bit-identical vs 2.2 (max|diff| = 0); sweep 5,287.5 s (88.1 min) wall / 8.8 GPU-h for 681 job-seeds — total job 1:31:47, inside the 2 h `gpu_debug` cap (target ~1.75 h).
7. **The honest answer to "is the mlp-2.0 architecture still meaningful?":** YES on capacity — the 2-regime MLP exceeds 0.80 as a single config for the first time in the 1.0–2.3 series, and the mixed family's val-selection signal is now positive. NO on 54/96 val-selection — the deployed rule still lands at 0.7596/0.7595, with the 3-layer anchor (54) and the big net (96) as the persistent val favorites. The val-year diagnostic localizes the remaining noise to the 54/96 val-2022 half.
